# Pipeline Runner

Each cell is **self-contained** — click any one to run that use case.

## Results

| Use Case | Run ID | Mode | Val Acc | Test Acc |
|----------|--------|------|---------|----------|
| Music | `20260322_171209_music` | Rule-based | 78.2% | 66.2% |
| Games | `20260322_175903_games` | Rule-based | 73.1% | 81.5% |
| Books | `20260322_155058_books` | Rule-based | 71.2% | 82.5% |
| Companies | `20260322_102925_companies` | Rule-based | 48.5% | 69.7% |
| Restaurant | `20260322_162759_restaurant` | Rule-based | 88.4% | 89.5% |


In [ ]:
# Games Datasets - Rule Based (Blocking+Matching pre-loaded)
import sys, os, json
from pathlib import Path

AGENTS = Path.cwd() if (Path.cwd() / "pipeline_agent.py").exists() else Path.cwd() / "agents"
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))
os.chdir(str(AGENTS))
from _resolve_import import ensure_project_root; ensure_project_root()

import config
from config import INPUT_DIR, GRAPH_RECURSION_LIMIT, LLM_REQUEST_TIMEOUT, OPENAI_MAX_RETRIES
from pipeline_agent import SimpleModelAgent, ProfileDatasetTool, SearchDocumentationTool
from required_logging import attach_logging
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", request_timeout=LLM_REQUEST_TIMEOUT, max_retries=OPENAI_MAX_RETRIES)
I = INPUT_DIR + "datasets/games/"
datasets = [I + "dbpedia.xml", I + "metacritic.xml", I + "sales.xml"]


BLOCKING_CONFIG = {"blocking_strategies":{"dbpedia_sales":{"strategy":"semantic_similarity","columns":["name","platform","releaseYear"],"params":{"top_k":20},"pair_completeness":0.912621359223301,"num_candidates":929573,"is_acceptable":True},"metacritic_dbpedia":{"strategy":"semantic_similarity","columns":["name","developer","platform"],"params":{"top_k":20},"pair_completeness":0.9625668449197861,"num_candidates":409850,"is_acceptable":True},"metacritic_sales":{"strategy":"semantic_similarity","columns":["name","platform","releaseYear"],"params":{"top_k":20},"pair_completeness":1.0,"num_candidates":409504,"is_acceptable":True}},"id_columns":{"dbpedia":"id","metacritic":"id","sales":"id"}}

MATCHING_CONFIG = {"id_columns":{"dbpedia":"id","metacritic":"id","sales":"id"},"matching_strategies":{"dbpedia_sales":{"comparators":[{"type":"string","column":"name","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"platform","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"developer","similarity_function":"cosine","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"date","column":"releaseYear","max_days_difference":365}],"weights":[0.45,0.2,0.15,0.2],"threshold":0.75,"f1":0.864367816091954},"metacritic_dbpedia":{"comparators":[{"type":"string","column":"name","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"platform","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"developer","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"date","column":"releaseYear","max_days_difference":366}],"weights":[0.45,0.2,0.2,0.15],"threshold":0.8,"f1":0.8866995073891625},"metacritic_sales":{"comparators":[{"type":"string","column":"name","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"platform","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"developer","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"numeric","column":"criticScore","max_difference":5.0},{"type":"numeric","column":"userScore","max_difference":1.0},{"type":"date","column":"releaseYear","max_days_difference":365}],"weights":[0.35,0.2,0.15,0.12,0.08,0.1],"threshold":0.78,"f1":0.8844884488448845}}}

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
agent = SimpleModelAgent(llm, tools={profile_tool.name: profile_tool, search_tool.name: search_tool})
attach_logging(agent, output_dir=config.OUTPUT_DIR, notebook_name="run_games_rb", use_case="games", llm_model="gpt-5.4")

print("[*] Starting games run (rule-based, blocking+matching pre-loaded)...")
result = agent.graph.invoke({
    "datasets": datasets,
    "original_datasets": list(datasets),
    "normalized_datasets": [],
    "entity_matching_testsets": {
        ("dbpedia", "sales"): I + "testsets/dbpedia_2_sales_test.csv",
        ("metacritic", "dbpedia"): I + "testsets/metacritic_2_dbpedia_test.csv",
        ("metacritic", "sales"): I + "testsets/metacritic_2_sales_test.csv",
    },
    "fusion_testset": I + "testsets/test_set_fusion.xml",
    "validation_fusion_testset": I + "testsets/validation_set_fusion.xml",
    "matcher_mode": "rulebased",
    "blocking_config": BLOCKING_CONFIG,
    "matching_config": MATCHING_CONFIG,
    "evaluation_attempts": 0,
    "normalization_attempts": 0,
    "normalization_execution_result": "",
    "normalization_rework_required": False,
    "normalization_rework_reasons": [],
    "normalization_directives": {},
    "investigator_decision": "",
}, config={"recursion_limit": GRAPH_RECURSION_LIMIT})

print("[*] Games run complete")
print(json.dumps({
    "case": "games",
    "val_acc": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
    "test_acc": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
    "iters": result.get("evaluation_attempts"),
    "run_id": result.get("run_id"),
}, indent=2, default=str))


[*] Starting games run (rule-based, blocking+matching pre-loaded)...
[SCHEMA] Aligning dataset schemas...
[*] Schema alignment columns:
    dbpedia: ['id', 'name', 'releaseYear', 'developer', 'platform', 'series']
    metacritic: ['id', 'name', 'releaseYear', 'developer', 'platform', 'criticScore', 'userScore', 'ESRB']
    sales: ['id', 'name', 'releaseYear', 'developer', 'publisher', 'platform', 'criticScore', 'userScore', 'ESRB', 'globalSales']
[PROFILE] Profiling datasets...
TOKEN USAGE (profile_data):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 310
   Completion tokens: 110
   Total tokens: 420
   Estimated cost: $0.002425
[NORM] Starting normalization (LLM-driven spec generation)
[NORM] Generating specs via LLM (attempt 1)
TOKEN USAGE (normalization_spec_generation):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 5,908
   Completion tokens: 259
   Total tokens: 6,167
   Estimated cost: $0.018655
[NORM] Specs for 3 dataset(s), 1 list column(s)
[NORM] Reasoning: Target stores rel

/Users/nickblazek/Desktop/Agent_Pipeline/agents/helpers/evaluation.py:142: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  fused_df = pd.read_csv(fused_path)


[EVAL DECISION] Attempt 1 | accuracy=41.250% | problems=3 problem(s) | investigating
  - overall_accuracy below target: 41.250% < 85.000%
  - low attribute accuracy: criticScore=20.000%, macro=41.250%, name=20.000%, releaseYear=20.000%, userScore=30.000%
  - low direct ID coverage: 10.000%


/Users/nickblazek/Desktop/Agent_Pipeline/agents/helpers/investigator_probe_runner.py:713: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  fused_df = pd.read_csv(fused_path)


[CLUSTER] Analyzing pairwise correspondences...
[CLUSTER] Analyzing 3 pair(s)...

[*] Analyzing: correspondences_dbpedia_metacritic.csv (27282 correspondences)
    Health: long_tail=NO, clusters=4428, max_size=1075
    One-to-many: 62.1%
    Scores: mean=0.871, IQR=0.087
    Ambiguity ratio: 71.1%
    Best strategy: MaximumBipartiteMatching (needs refinement)
    All recommended: MaximumBipartiteMatching, StableMatching, HierarchicalClusterer
[*] Analyzing: correspondences_dbpedia_sales.csv (20730 correspondences)
    Health: long_tail=NO, clusters=2535, max_size=1008
    One-to-many: 66.2%
    Scores: mean=0.838, IQR=0.111
    Ambiguity ratio: 66.6%
    Best strategy: MaximumBipartiteMatching (needs refinement)
    All recommended: MaximumBipartiteMatching, StableMatching, HierarchicalClusterer
[*] Analyzing: correspondences_metacritic_sales.csv (11072 correspondences)
    Health: long_tail=NO, clusters=5416, max_size=92
    One-to-many: 34.2%
    Scores: mean=0.908, IQR=0.151
    Amb

...

[CLUSTER] Analyzing pairwise correspondences...
[CLUSTER] Analyzing 3 pair(s)...

[*] Analyzing: correspondences_dbpedia_metacritic.csv (27282 correspondences)
    Health: long_tail=NO, clusters=4428, max_size=1075
    One-to-many: 62.1%
    Scores: mean=0.871, IQR=0.087
    Ambiguity ratio: 71.1%
    Best strategy: MaximumBipartiteMatching (needs refinement)
    All recommended: MaximumBipartiteMatching, StableMatching, HierarchicalClusterer
[*] Analyzing: correspondences_dbpedia_sales.csv (20730 correspondences)
    Health: long_tail=NO, clusters=2535, max_size=1008
    One-to-many: 66.2%
    Scores: mean=0.838, IQR=0.111
    Ambiguity ratio: 66.6%
    Best strategy: MaximumBipartiteMatching (needs refinement)
    All recommended: MaximumBipartiteMatching, StableMatching, HierarchicalClusterer
[*] Analyzing: correspondences_metacritic_sales.csv (11072 correspondences)
    Health: long_tail=NO, clusters=5416, max_size=92
    One-to-many: 34.2%
    Scores: mean=0.908, IQR=0.151
    Amb

/Users/nickblazek/Desktop/Agent_Pipeline/agents/helpers/evaluation.py:142: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  fused_df = pd.read_csv(fused_path)


[EVAL DECISION] Attempt 3 | accuracy=86.250% | problems=1 problem(s) | quality gate reached
  - high missing_fused_value mismatch ratio: 27.273%
TOTAL TOKEN USAGE:
   Prompt tokens: 231,358
   Completion tokens: 22,366
   Total tokens: 253,724
   Estimated cost: $0.913885


/Users/nickblazek/Desktop/Agent_Pipeline/agents/helpers/investigator_probe_runner.py:713: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  fused_df = pd.read_csv(fused_path)


[INVESTIGATION] Early exit → human_review_export (accuracy=86.2%, attempt 3/4)
[REVIEW] Building human-review package...
TOKEN USAGE (human_review_export_generation):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 1,280
   Completion tokens: 5,018
   Total tokens: 6,298
   Estimated cost: $0.078470
[REVIEW] Human-review package complete
[SEALED TEST] Evaluating on held-out test set...
[EVAL CODE] Generating evaluation script...
TOKEN USAGE (evaluation_adaption):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 12,105
   Completion tokens: 130
   Total tokens: 12,235
   Estimated cost: $0.032212
[SEARCH TOOL CALL 1]: Asking 'How to use PyDI DataFusionEvaluator evaluate with fused_df, gold_df, strategy, and evaluation functions like tokenized_match, year_only_match, numeric_tolerance_match, set_equality_match, exact_match, boolean_match?'
[SEARCH TOOL CALL 2]: Asking 'PyDI evaluation functions for data fusion: tokenized_match thresholds, year_only_match behavior, numeric_tolerance_match to

In [ ]:
#Music. Datasets Rule Based run (with blocking+matching pre-loaded) 
import sys; sys.dont_write_bytecode = True
import json, os
from pathlib import Path
import importlib, sys; [sys.modules.pop(k) for k in list(sys.modules) if any(x in k for x in ['helpers', 'pipeline_agent', 'code_guardrails', 'prompts', 'config', 'required_logging', 'workflow_logging', '_resolve_import'])]


AGENTS = Path.cwd() if (Path.cwd() / "pipeline_agent.py").exists() else Path.cwd() / "agents"
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))
os.chdir(str(AGENTS))

# Force-reload all project modules (in case kernel has stale imports)
[sys.modules.pop(k) for k in list(sys.modules) if any(x in k for x in ['helpers', 'pipeline_agent', 'code_guardrails', 'prompts', 'config', 'required_logging', 'workflow_logging', '_resolve_import'])]

from _resolve_import import ensure_project_root; ensure_project_root()
import config
from config import INPUT_DIR, GRAPH_RECURSION_LIMIT, LLM_REQUEST_TIMEOUT, OPENAI_MAX_RETRIES
from pipeline_agent import SimpleModelAgent, ProfileDatasetTool, SearchDocumentationTool
from required_logging import attach_logging
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", request_timeout=LLM_REQUEST_TIMEOUT, max_retries=3)
I = INPUT_DIR + "datasets/music/"
datasets = [I + "discogs.xml", I + "lastfm.xml", I + "musicbrainz.xml"]

BLOCKING_CONFIG = {
    "blocking_strategies": {
        "discogs_lastfm": {
            "strategy": "semantic_similarity",
            "columns": ["name", "artist"],
            "params": {"top_k": 10},
            "pair_completeness": 0.9642857142857143,
            "num_candidates": 225831,
            "is_acceptable": True,
        },
        "discogs_musicbrainz": {
            "strategy": "sorted_neighbourhood",
            "columns": ["name"],
            "params": {"window": 15},
            "pair_completeness": 0.9,
            "num_candidates": 115280,
            "is_acceptable": True,
        },
        "musicbrainz_lastfm": {
            "strategy": "semantic_similarity",
            "columns": ["name", "artist", "duration"],
            "params": {"top_k": 10},
            "pair_completeness": 0.9240506329113924,
            "num_candidates": 47615,
            "is_acceptable": True,
        },
    },
    "id_columns": {"discogs": "id", "lastfm": "id", "musicbrainz": "id"},
}

MATCHING_CONFIG = {
    "id_columns": {"discogs": "id", "lastfm": "id", "musicbrainz": "id"},
    "matching_strategies": {
        "discogs_lastfm": {
            "comparators": [
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "name", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "tracks_track_name", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "set_jaccard"},
                {"type": "numeric", "column": "duration", "max_difference": 10.0},
            ],
            "weights": [0.3, 0.35, 0.25, 0.1],
            "threshold": 0.6,
            "f1": 0.896551724137931,
        },
        "discogs_musicbrainz": {
            "comparators": [
                {"type": "string", "column": "name", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "date", "column": "release-date", "max_days_difference": 365},
                {"type": "string", "column": "release-country", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "numeric", "column": "duration", "max_difference": 60.0},
            ],
            "weights": [0.35, 0.25, 0.2, 0.1, 0.1],
            "threshold": 0.7,
            "f1": 0.8732394366197184,
        },
        "musicbrainz_lastfm": {
            "comparators": [
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "name", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "duration", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
            ],
            "weights": [0.35, 0.45, 0.2],
            "threshold": 0.75,
            "f1": 0.8652482269503546,
        },
    },
}

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
agent = SimpleModelAgent(llm, tools={profile_tool.name: profile_tool, search_tool.name: search_tool})
attach_logging(agent, output_dir=config.OUTPUT_DIR, notebook_name="pipeline_runner", use_case="music", llm_model="gpt-5.4")

print("[*] Starting music run (blocking+matching pre-loaded)...")
result = agent.graph.invoke({
    "datasets": datasets,
    "original_datasets": list(datasets),
    "normalized_datasets": [],
    "entity_matching_testsets": {
        ("discogs", "lastfm"): I + "testsets/discogs_lastfm_goldstandard_blocking.csv",
        ("discogs", "musicbrainz"): I + "testsets/discogs_musicbrainz_goldstandard_blocking.csv",
        ("musicbrainz", "lastfm"): I + "testsets/musicbrainz_lastfm_goldstandard_blocking.csv",
    },
    "fusion_testset": I + "testsets/test_set.xml",
    "validation_fusion_testset": I + "testsets/validation_set.xml",
    "matcher_mode": "rulebased",
    "blocking_config": BLOCKING_CONFIG,
    "matching_config": MATCHING_CONFIG,
    "evaluation_attempts": 0,
    "normalization_attempts": 0,
    "normalization_execution_result": "",
    "normalization_rework_required": False,
    "normalization_rework_reasons": [],
    "normalization_directives": {},
    "investigator_decision": "",
}, config={"recursion_limit": GRAPH_RECURSION_LIMIT})

print("[*] Music run complete")
print(json.dumps({
    "case": "music",
    "val_acc": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
    "test_acc": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
    "iters": result.get("evaluation_attempts"),
    "run_id": result.get("run_id"),
}, indent=2, default=str))


[*] Starting music run (blocking+matching pre-loaded)...
[SCHEMA] Aligning dataset schemas...
[*] Schema alignment columns:
    discogs: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'label', 'genre', 'tracks_track_name', 'tracks_track_position', 'tracks_track_duration']
    lastfm: ['id', 'name', 'artist', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
    musicbrainz: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
[PROFILE] Profiling datasets...
TOKEN USAGE (profile_data):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 312
   Completion tokens: 112
   Total tokens: 424
   Estimated cost: $0.002460
[NORM] Starting normalization (LLM-driven spec generation)
[NORM] Generating specs via LLM (attempt 1)
TOKEN USAGE (normalization_spec_generation):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 16,337
   Completion tokens: 260
   Tota

In [ ]:
# Educational Music Rule Based run (with blocking+matching pre-loaded) and val = test set
import sys; sys.dont_write_bytecode = True
import json, os
from pathlib import Path
import importlib, sys; [sys.modules.pop(k) for k in list(sys.modules) if any(x in k for x in ['helpers', 'pipeline_agent', 'code_guardrails', 'prompts', 'config', 'required_logging', 'workflow_logging', '_resolve_import'])]


AGENTS = Path.cwd() if (Path.cwd() / "pipeline_agent.py").exists() else Path.cwd() / "agents"
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))
os.chdir(str(AGENTS))

# Force-reload all project modules (in case kernel has stale imports)
[sys.modules.pop(k) for k in list(sys.modules) if any(x in k for x in ['helpers', 'pipeline_agent', 'code_guardrails', 'prompts', 'config', 'required_logging', 'workflow_logging', '_resolve_import'])]

from _resolve_import import ensure_project_root; ensure_project_root()
import config
from config import INPUT_DIR, GRAPH_RECURSION_LIMIT, LLM_REQUEST_TIMEOUT, OPENAI_MAX_RETRIES
from pipeline_agent import SimpleModelAgent, ProfileDatasetTool, SearchDocumentationTool
from required_logging import attach_logging
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", request_timeout=LLM_REQUEST_TIMEOUT, max_retries=3)
I = INPUT_DIR + "datasets/music/"
datasets = [I + "discogs.xml", I + "lastfm.xml", I + "musicbrainz.xml"]

BLOCKING_CONFIG = {
    "blocking_strategies": {
        "discogs_lastfm": {
            "strategy": "semantic_similarity",
            "columns": ["name", "artist"],
            "params": {"top_k": 10},
            "pair_completeness": 0.9642857142857143,
            "num_candidates": 225831,
            "is_acceptable": True,
        },
        "discogs_musicbrainz": {
            "strategy": "sorted_neighbourhood",
            "columns": ["name"],
            "params": {"window": 15},
            "pair_completeness": 0.9,
            "num_candidates": 115280,
            "is_acceptable": True,
        },
        "musicbrainz_lastfm": {
            "strategy": "semantic_similarity",
            "columns": ["name", "artist", "duration"],
            "params": {"top_k": 10},
            "pair_completeness": 0.9240506329113924,
            "num_candidates": 47615,
            "is_acceptable": True,
        },
    },
    "id_columns": {"discogs": "id", "lastfm": "id", "musicbrainz": "id"},
}

MATCHING_CONFIG = {
    "id_columns": {"discogs": "id", "lastfm": "id", "musicbrainz": "id"},
    "matching_strategies": {
        "discogs_lastfm": {
            "comparators": [
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "name", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "tracks_track_name", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "set_jaccard"},
                {"type": "numeric", "column": "duration", "max_difference": 10.0},
            ],
            "weights": [0.3, 0.35, 0.25, 0.1],
            "threshold": 0.6,
            "f1": 0.896551724137931,
        },
        "discogs_musicbrainz": {
            "comparators": [
                {"type": "string", "column": "name", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "date", "column": "release-date", "max_days_difference": 365},
                {"type": "string", "column": "release-country", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "numeric", "column": "duration", "max_difference": 60.0},
            ],
            "weights": [0.35, 0.25, 0.2, 0.1, 0.1],
            "threshold": 0.7,
            "f1": 0.8732394366197184,
        },
        "musicbrainz_lastfm": {
            "comparators": [
                {"type": "string", "column": "artist", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "name", "similarity_function": "cosine", "preprocess": "lower_strip", "list_strategy": "concatenate"},
                {"type": "string", "column": "duration", "similarity_function": "jaro_winkler", "preprocess": "lower_strip", "list_strategy": "concatenate"},
            ],
            "weights": [0.35, 0.45, 0.2],
            "threshold": 0.75,
            "f1": 0.8652482269503546,
        },
    },
}

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
agent = SimpleModelAgent(llm, tools={profile_tool.name: profile_tool, search_tool.name: search_tool})
attach_logging(agent, output_dir=config.OUTPUT_DIR, notebook_name="pipeline_runner", use_case="music", llm_model="gpt-5.4")

print("[*] Starting music run (blocking+matching pre-loaded)...")
result = agent.graph.invoke({
    "datasets": datasets,
    "original_datasets": list(datasets),
    "normalized_datasets": [],
    "entity_matching_testsets": {
        ("discogs", "lastfm"): I + "testsets/discogs_lastfm_goldstandard_blocking.csv",
        ("discogs", "musicbrainz"): I + "testsets/discogs_musicbrainz_goldstandard_blocking.csv",
        ("musicbrainz", "lastfm"): I + "testsets/musicbrainz_lastfm_goldstandard_blocking.csv",
    },
    "validation_fusion_testset": I + "testsets/test_set.xml",
    "matcher_mode": "rulebased",
    "blocking_config": BLOCKING_CONFIG,
    "matching_config": MATCHING_CONFIG,
    "evaluation_attempts": 0,
    "normalization_attempts": 0,
    "normalization_execution_result": "",
    "normalization_rework_required": False,
    "normalization_rework_reasons": [],
    "normalization_directives": {},
    "investigator_decision": "",
}, config={"recursion_limit": GRAPH_RECURSION_LIMIT})

print("[*] Music run complete")
print(json.dumps({
    "case": "music",
    "val_acc": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
    "test_acc": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
    "iters": result.get("evaluation_attempts"),
    "run_id": result.get("run_id"),
}, indent=2, default=str))


/Users/nickblazek/Desktop/Agent_Pipeline/agents/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Starting music run (blocking+matching pre-loaded)...
[SCHEMA] Aligning dataset schemas...
[*] Schema alignment columns:
    discogs: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'label', 'genre', 'tracks_track_name', 'tracks_track_position', 'tracks_track_duration']
    lastfm: ['id', 'name', 'artist', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
    musicbrainz: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
[PROFILE] Profiling datasets...
TOKEN USAGE (profile_data):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 312
   Completion tokens: 112
   Total tokens: 424
   Estimated cost: $0.002460
[NORM] Starting normalization (LLM-driven spec generation)
[NORM] Generating specs via LLM (attempt 1)
TOKEN USAGE (normalization_spec_generation):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 14,364
   Completion tokens: 401
   Tota

/Users/nickblazek/Desktop/Agent_Pipeline/agents/pipeline_agent.py:279: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)


[SEARCH TOOL CALL 2]: Asking 'PyDI load_xml nested_handling aggregate usage and DataFusionStrategy add_evaluation_function examples for evaluation.'
TOKEN USAGE (evaluation_adaption):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 16,669
   Completion tokens: 3,050
   Total tokens: 19,719
   Estimated cost: $0.087423
[EVAL EXEC] Running evaluation (attempt 1)...
[EVAL EXEC] Evaluation completed successfully
[EVAL] Evaluation succeeded
[INVESTIGATION] Starting investigator node
[EVAL DECISION] Processing evaluation metrics...
[EVAL DECISION] New best accuracy: 0.6395939086294417
[EVAL DECISION] Attempt 1 | accuracy=63.959% | problems=4 problem(s) | investigating
  - overall_accuracy below target: 63.959% < 85.000%
  - low attribute accuracy: label=37.500%, tracks_track_duration=40.000%, tracks_track_name=47.826%
  - low direct ID coverage: 30.435%
  - high missing_fused_value mismatch ratio: 63.380%
[CLUSTER] Analyzing pairwise correspondences...
[CLUSTER] Analyzing 3 pair(s)...

[*] An

In [ ]:
# Restaurants Datasets - Rule Based (Blocking+Matching pre-loaded)
import sys, os, json
from pathlib import Path
AGENTS = Path.cwd() if (Path.cwd() / "pipeline_agent.py").exists() else Path.cwd() / "agents"
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))
os.chdir(str(AGENTS))
from _resolve_import import ensure_project_root; ensure_project_root()
import config
from config import INPUT_DIR, GRAPH_RECURSION_LIMIT, LLM_REQUEST_TIMEOUT, OPENAI_MAX_RETRIES
from pipeline_agent import SimpleModelAgent, ProfileDatasetTool, SearchDocumentationTool
from required_logging import attach_logging
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5.4", request_timeout=LLM_REQUEST_TIMEOUT, max_retries=OPENAI_MAX_RETRIES)
I = INPUT_DIR + "datasets/restaurant/"
datasets = [I + "kaggle_small.parquet", I + "uber_eats_small.parquet", I + "yelp_small.parquet"]

BLOCKING_CONFIG = {"blocking_strategies":{"kaggle_small_uber_eats_small":{"strategy":"semantic_similarity","columns":["name_norm","city","state"],"params":{"top_k":20},"pair_completeness":1.0,"num_candidates":199719,"is_acceptable":True},"kaggle_small_yelp_small":{"strategy":"exact_match_multi","columns":["phone_e164","postal_code"],"params":{},"pair_completeness":1.0,"num_candidates":179,"is_acceptable":True},"uber_eats_small_yelp_small":{"strategy":"semantic_similarity","columns":["name_norm","street","city"],"params":{"top_k":20},"pair_completeness":1.0,"num_candidates":199972,"is_acceptable":True}},"id_columns":{"kaggle_small":"id","uber_eats_small":"id","yelp_small":"id"}}

MATCHING_CONFIG = {"id_columns":{"kaggle_small":"id","uber_eats_small":"id","yelp_small":"id"},"matching_strategies":{"kaggle_small_uber_eats_small":{"comparators":[{"type":"string","column":"name_norm","similarity_function":"cosine","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"street","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"city","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"state","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"numeric","column":"latitude","max_difference":0.01},{"type":"numeric","column":"longitude","max_difference":0.01},{"type":"string","column":"categories","similarity_function":"jaccard","preprocess":"lower_strip","list_strategy":"set_jaccard"}],"weights":[0.35,0.12,0.08,0.05,0.16,0.16,0.08],"threshold":0.75,"f1":0.8947368421052632},"kaggle_small_yelp_small":{"comparators":[{"type":"string","column":"phone_e164","similarity_function":"jaro_winkler","list_strategy":"concatenate"},{"type":"string","column":"name_norm","similarity_function":"cosine","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"postal_code","similarity_function":"jaro_winkler","preprocess":"strip","list_strategy":"concatenate"},{"type":"string","column":"city","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"numeric","column":"latitude","max_difference":0.002},{"type":"numeric","column":"longitude","max_difference":0.002},{"type":"string","column":"categories","similarity_function":"jaccard","preprocess":"lower_strip","list_strategy":"set_jaccard"}],"weights":[0.2,0.25,0.15,0.1,0.1,0.1,0.1],"threshold":0.75,"f1":0.983050847457627},"uber_eats_small_yelp_small":{"comparators":[{"type":"string","column":"name_norm","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"street","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"city","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"numeric","column":"latitude","max_difference":0.002},{"type":"numeric","column":"longitude","max_difference":0.002},{"type":"string","column":"categories","similarity_function":"jaccard","preprocess":"lower_strip","list_strategy":"set_jaccard"}],"weights":[0.28,0.16,0.12,0.16,0.16,0.12],"threshold":0.72,"f1":0.9302325581395349}}}

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
agent = SimpleModelAgent(llm, tools={profile_tool.name: profile_tool, search_tool.name: search_tool})
attach_logging(agent, output_dir=config.OUTPUT_DIR, notebook_name="run_restaurant_rb", use_case="restaurant", llm_model="gpt-5.4")
print("[*] Starting restaurant run (rule-based, blocking+matching pre-loaded)...")
result = agent.graph.invoke({
"datasets": datasets,
"original_datasets": list(datasets),
"normalized_datasets": [],
"entity_matching_testsets": {
        ("kaggle_small", "uber_eats_small"): I + "testsets/kaggle_uber_eats_goldstandard_blocking_small.csv",
        ("kaggle_small", "yelp_small"): I + "testsets/kaggle_yelp_goldstandard_blocking_small.csv",
        ("uber_eats_small", "yelp_small"): I + "testsets/uber_eats_yelp_goldstandard_blocking_small.csv",
    },
"fusion_testset": I + "testsets/Restaurant_Fusion_Test_Set.csv",
"validation_fusion_testset": I + "testsets/Restaurant_Fusion_Validation_Set.csv",
"matcher_mode": "rulebased",
"blocking_config": BLOCKING_CONFIG,
"matching_config": MATCHING_CONFIG,
"evaluation_attempts": 0,
"normalization_attempts": 0,
"normalization_execution_result": "",
"normalization_rework_required": False,
"normalization_rework_reasons": [],
"normalization_directives": {},
"investigator_decision": "",
}, config={"recursion_limit": GRAPH_RECURSION_LIMIT})
print("[*] Restaurant run complete")
print(json.dumps({
"case": "restaurant",
"val_acc": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
"test_acc": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
"iters": result.get("evaluation_attempts"),
"run_id": result.get("run_id"),
}, indent=2, default=str))


/Users/nickblazek/Desktop/Agent_Pipeline/agents/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Starting restaurant run (rule-based, blocking+matching pre-loaded)...
[SCHEMA] Aligning dataset schemas...
[*] Schema alignment columns:
    kaggle_small: ['id', 'source', 'name', 'name_norm', 'website', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating']
    uber_eats_small: ['id', 'source', 'name', 'name_norm', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']
    yelp_small: ['id', 'source', 'name', 'name_norm', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']
[PROFILE] Profiling datasets...
TOKEN USAGE (profile_data):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 321
   C

/Users/nickblazek/Desktop/Agent_Pipeline/agents/pipeline_agent.py:279: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)


TOKEN USAGE (evaluation_adaption):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 17,835
   Completion tokens: 3,031
   Total tokens: 20,866
   Estimated cost: $0.090052
[EVAL EXEC] Running evaluation (attempt 1)...
[EVAL EXEC] Evaluation completed successfully
[EVAL] Evaluation succeeded
[INVESTIGATION] Starting investigator node
[EVAL DECISION] Processing evaluation metrics...
[EVAL DECISION] New best accuracy: 0.265
[EVAL DECISION] Attempt 1 | accuracy=26.500% | problems=4 problem(s) | investigating
  - overall_accuracy below target: 26.500% < 85.000%
  - low attribute accuracy: _id=30.000%, address_line1=25.000%, categories=30.000%, city=30.000%, country=30.000%, house_number=30.000%, latitude=30.000%, longitude=30.000%
  - low direct ID coverage: 30.000%
  - high missing_fused_value mismatch ratio: 94.218%
[CLUSTER] Analyzing pairwise correspondences...
[CLUSTER] Analyzing 3 pair(s)...

[*] Analyzing: correspondences_kaggle_small_uber_eats_small.csv (58 correspondences)
    Health

In [ ]:
import sys, os, json
from pathlib import Path
AGENTS = Path.cwd() if (Path.cwd() / "pipeline_agent.py").exists() else Path.cwd() / "agents"
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))
os.chdir(str(AGENTS))
from _resolve_import import ensure_project_root; ensure_project_root()
import config
from config import INPUT_DIR, GRAPH_RECURSION_LIMIT, LLM_REQUEST_TIMEOUT, OPENAI_MAX_RETRIES
from pipeline_agent import SimpleModelAgent, ProfileDatasetTool, SearchDocumentationTool
from required_logging import attach_logging
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5.4", request_timeout=LLM_REQUEST_TIMEOUT, max_retries=OPENAI_MAX_RETRIES)
I = INPUT_DIR + "datasets/books/"
datasets = [I + "amazon_small.parquet", I + "goodreads_small.parquet", I + "metabooks_small.parquet"]

BLOCKING_CONFIG = {"blocking_strategies":{"goodreads_small_amazon_small":{"strategy":"semantic_similarity","columns":["title","author","publish_year"],"params":{"top_k":20},"pair_completeness":0.9977578475336323,"num_candidates":199823,"is_acceptable":True},"metabooks_small_amazon_small":{"strategy":"semantic_similarity","columns":["title","author","publisher"],"params":{"top_k":15},"pair_completeness":0.9939577039274925,"num_candidates":149824,"is_acceptable":True},"metabooks_small_goodreads_small":{"strategy":"semantic_similarity","columns":["title","author","publisher"],"params":{"top_k":20},"pair_completeness":0.9947826086956522,"num_candidates":199106,"is_acceptable":True}},"id_columns":{"amazon_small":"id","goodreads_small":"id","metabooks_small":"id"}}

MATCHING_CONFIG = {"id_columns":{"amazon_small":"id","goodreads_small":"id","metabooks_small":"id"},"matching_strategies":{"goodreads_small_amazon_small":{"comparators":[{"type":"string","column":"title","similarity_function":"jaro_winkler","preprocess":"lower_strip"},{"type":"string","column":"author","similarity_function":"jaro_winkler","preprocess":"lower_strip"},{"type":"numeric","column":"publish_year","max_difference":1.0},{"type":"string","column":"publisher","similarity_function":"cosine","preprocess":"lower_strip"}],"weights":[0.42,0.33,0.15,0.1],"threshold":0.66,"f1":0.8029330889092575},"metabooks_small_amazon_small":{"comparators":[{"type":"string","column":"title","similarity_function":"cosine","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"author","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"string","column":"publisher","similarity_function":"jaro_winkler","preprocess":"lower_strip","list_strategy":"concatenate"},{"type":"numeric","column":"publish_year","max_difference":1.0}],"weights":[0.5,0.25,0.15,0.1],"threshold":0.72,"f1":0.852589641434263},"metabooks_small_goodreads_small":{"comparators":[{"type":"string","column":"title","similarity_function":"jaro_winkler","preprocess":"lower_strip"},{"type":"string","column":"author","similarity_function":"jaro_winkler","preprocess":"lower_strip"},{"type":"string","column":"publisher","similarity_function":"jaro_winkler","preprocess":"lower_strip"},{"type":"string","column":"genres","similarity_function":"jaccard","preprocess":"lower_strip","list_strategy":"set_jaccard"},{"type":"numeric","column":"publish_year","max_difference":1.0}],"weights":[0.42,0.23,0.12,0.15,0.08],"threshold":0.68,"f1":0.8362676056338028}}}

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
agent = SimpleModelAgent(llm, tools={profile_tool.name: profile_tool, search_tool.name: search_tool})
attach_logging(agent, output_dir=config.OUTPUT_DIR, notebook_name="run_books_rb", use_case="books", llm_model="gpt-5.4")
print("[*] Starting books run (rule-based, blocking+matching pre-loaded)...")
result = agent.graph.invoke({
"datasets": datasets,
"original_datasets": list(datasets),
"normalized_datasets": [],
"entity_matching_testsets": {
        ("goodreads_small", "amazon_small"): I + "testsets/goodreads_2_amazon.csv",
        ("metabooks_small", "amazon_small"): I + "testsets/metabooks_2_amazon.csv",
        ("metabooks_small", "goodreads_small"): I + "testsets/metabooks_2_goodreads.csv",
    },
"fusion_testset": I + "testsets/test_set.csv",
"validation_fusion_testset": I + "testsets/validation_set.csv",
"matcher_mode": "rulebased",
"blocking_config": BLOCKING_CONFIG,
"matching_config": MATCHING_CONFIG,
"evaluation_attempts": 0,
"normalization_attempts": 0,
"normalization_execution_result": "",
"normalization_rework_required": False,
"normalization_rework_reasons": [],
"normalization_directives": {},
"investigator_decision": "",
}, config={"recursion_limit": GRAPH_RECURSION_LIMIT})
print("[*] Books run complete")
print(json.dumps({
"case": "books",
"val_acc": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
"test_acc": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
"iters": result.get("evaluation_attempts"),
"run_id": result.get("run_id"),
}, indent=2, default=str))


/Users/nickblazek/Desktop/Agent_Pipeline/agents/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Starting books run (rule-based, blocking+matching pre-loaded)...
[SCHEMA] Aligning dataset schemas...
[*] Schema alignment columns:
    amazon_small: ['id', 'title', 'author', 'publish_year', 'publisher', 'isbn_clean']
    goodreads_small: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'bookformat', 'edition', 'page_count', 'publisher', 'publish_year', 'price', 'isbn_clean']
    metabooks_small: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean']
[PROFILE] Profiling datasets...
TOKEN USAGE (profile_data):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 317
   Completion tokens: 117
   Total tokens: 434
   Estimated cost: $0.002547
[NORM] Starting normalization (LLM-driven spec generation)
[NORM] Generating specs via LLM (attempt 1)
TOKEN USAGE (normalization_spec_generation):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 8,715
   Completion tokens: 258
   Total tokens: 8,9

/Users/nickblazek/Desktop/Agent_Pipeline/agents/pipeline_agent.py:279: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)


TOKEN USAGE (evaluation_adaption):
   Model: gpt-5.4-2026-03-05
   Prompt tokens: 13,189
   Completion tokens: 2,797
   Total tokens: 15,986
   Estimated cost: $0.074927
[EVAL EXEC] Running evaluation (attempt 1)...
[EVAL EXEC] Evaluation completed successfully
[EVAL] Evaluation succeeded
[INVESTIGATION] Starting investigator node
[EVAL DECISION] Processing evaluation metrics...
[EVAL DECISION] New best accuracy: 0.675
[EVAL DECISION] Attempt 1 | accuracy=67.500% | problems=2 problem(s) | investigating
  - overall_accuracy below target: 67.500% < 85.000%
  - low attribute accuracy: genres=0.000%
[CLUSTER] Analyzing pairwise correspondences...
[CLUSTER] Analyzing 3 pair(s)...

[*] Analyzing: correspondences_amazon_small_goodreads_small.csv (1902 correspondences)
    Health: long_tail=NO, clusters=1151, max_size=26
    One-to-many: 22.6%
    Scores: mean=0.769, IQR=0.157
    Ambiguity ratio: 57.1%
    Best strategy: MaximumBipartiteMatching (needs refinement)
    All recommended: Maximum